In [ ]:
import os
from pathlib import Path
import numpy as np
from sklearn.metrics import precision_recall_curve

# knows how to process the data
from anomalib.data import MVTecAD
# knows how to pass the image data through the Patchcore/ResNet-18 model (pre-trained)
# it is more like the manager
from anomalib.engine import Engine
# does the heavy lifting
# doesn't change the weights, instead extracts internal feature patches and stores them
# the stored feature patches allows it to remember what a "good" feature looks like
from anomalib.models import Patchcore

# 0.
# Datasets in our example data: mvtec_ad/bottle
# train/good/
# test/{broken_large,broken_small,contamination,good}
# ground_truth/{broken_large,broken_small,contamination}

# 1.
# Initialize the chosen mvtec_ad bottle dataset
datamodule = MVTecAD(root="../../data/raw/mvtec_ad", category="bottle")
# Initialize the model and model pipeline
# Patchcore is an algorithm designed for unsupervised industrial anomaly detection and localization
# - data/external/aupimo_benchmarks/patchcore_wr50 used it - but we are using a smaller model here
# - I've read a bit about the internals of it and how it splits open ResNet-18 for its application - but I do not have a full understanding yet
# Microsoft's ResNet-18 is an 18 layer Residual Network a specific CNN - the trick it uses allowed CNNs to become deeper (more layers)
# - don't ask me about the details, it was available, and with 18 layers small enough so I could run it on my machine (with two crashes, but one success)
# - used here as feature extractor
model = Patchcore(backbone="resnet18")
# Initialize the engine
engine = Engine(accelerator="gpu", devices=1, enable_progress_bar=False, enable_model_summary=False)

# 2. Run inference over your test/validation sets 
# train_test_split is done by the engine, not by a guessed 20/80 ratio, but on concrete data properties
engine.fit(model, datamodule)
# the engine then computes our predictions with the trained model
predictions = engine.predict(model=model, datamodule=datamodule)

# 3. Collect the true targets and raw pixel predictions
# data preparation step: deep learning tensors to 
all_preds = []
all_masks = []

for batch in predictions:
    # Extract anomaly maps and truth masks, flattening them for pixel-level PR computation
    all_preds.append(batch.anomaly_map.cpu().numpy().flatten())
    all_masks.append(batch.gt_mask.cpu().numpy().flatten())

y_scores = np.concatenate(all_preds)
y_true = np.concatenate(all_masks)

# 4. Generate every real coordinate point on the curve
precisions, recalls, thresholds = precision_recall_curve(y_true, y_scores)

# 5. Save the real arrays to disk
target_dir = Path("../../data/external/aupimo_benchmarks/patchcore_wr50/mvtec/bottle/")
os.makedirs(target_dir, exist_ok=True)

np.savez_compressed(
    target_dir / "real_pr_curve.npz", 
    precision=precisions, 
    recall=recalls, 
    thresholds=thresholds
)
print(f"Successfully generated and saved real curve coordinates to {target_dir / 'real_pr_curve.npz'}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/benni/Documents/antigravity_workspace/industrial-component-anomaly-detection/.pixi/envs/dev/lib/python3.12/site-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer
/home/benni/Documents/antigravity_workspace/industrial-component-anomaly-detection/.pixi/envs/dev/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:538: Found 69 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.
